# Practice Lab: Production Self-Hosting with vLLM on GKE (Lesson 13.6)

Module 13 . 4 exercises . Deploy + load test + hybrid routing + multi-model


# Lesson 13.6: Production Self-Hosting with vLLM on GKE

4 exercises: 1E + 2M + 1C

Module 13: Cost Optimization & Efficiency


In [ ]:
import random, numpy as np
random.seed(42); np.random.seed(42)


---
## Exercise 1 (Easy): Deploy & Test


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random, numpy as np
random.seed(42); np.random.seed(42)

class VS:
    def __init__(self,tps=80,mc=50,ghr=0.50): self.tps=tps; self.mc=mc; self.ghr=ghr
    def bench(self,n=10,ot=200,conc=10):
        lats=[]
        for _ in range(n):
            etps=self.tps*min(conc,self.mc)/max(conc,1)
            bl=ot/etps*1000; qd=max(0,(conc-self.mc))*200
            lat=max(100,round(bl+qd+abs(random.gauss(0,bl*0.1))))
            lats.append(lat)
        ls=sorted(lats); tt=sum(lats)/1000/conc
        gpu=min(95,conc/self.mc*100); cpr=self.ghr/3600*(sum(lats)/1000/n)
        return {"p50":ls[len(ls)//2],"p99":ls[-1],"rps":round(n/max(tt,0.001),1),
            "gpu":round(gpu,1),"cpr":round(cpr,6)}

v=VS()
print("vLLM Benchmark:")
print(f"  {'Conc':>6} {'P50':>6} {'P99':>6} {'RPS':>6} {'GPU%':>6} {'$/req':>10}")
for c in [1,5,10,25,50,80]:
    r=v.bench(max(c,10),200,c)
    print(f"  {c:>6} {r['p50']:>5}ms {r['p99']:>5}ms {r['rps']:>5.1f} {r['gpu']:>5.1f}% ${r['cpr']:.6f}")
mg=0.50*24*30; mr=v.bench(10,200,1)['rps']*86400*30
print(f"\n  Monthly: ${mg:.0f} GPU | {mr:,.0f} req capacity | ${mg/mr:.6f}/req")


</details>


---
## Exercise 2 (Medium): Load Test Scaling


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class HPA:
    def __init__(self,mn=1,mx=4,tg=70,tps=80,mc=50):
        self.mn=mn;self.mx=mx;self.tg=tg;self.tps=tps;self.mc=mc;self.pods=mn;self.cd=0;self.hist=[]
    def step(self,users):
        cap=self.pods*self.mc; gpu=min(95,users/cap*100)
        if users<=cap:
            p95=round(200/(self.tps*self.pods)*1000*(1+gpu/200)+random.gauss(0,100))
        else:
            p95=round(200/(self.tps*self.pods)*1000*(users/cap)*2+random.gauss(0,200))
        p95=max(200,p95); act="none"
        if self.cd>0: self.cd-=1
        elif gpu>self.tg and self.pods<self.mx: old=self.pods; self.pods+=1; self.cd=2; act=f"UP {old}->{self.pods}"
        elif gpu<40 and self.pods>self.mn: old=self.pods; self.pods-=1; self.cd=3; act=f"DN {old}->{self.pods}"
        self.hist.append({"u":users,"p":self.pods,"g":round(gpu,1),"p95":p95,"act":act})

h=HPA()
print("HPA Load Test:")
print(f"  {'Users':>6} {'Pods':>5} {'GPU%':>6} {'P95':>7} {'Action'}")
for u in [10,20,30,40,50,60,70,80,90,100]:
    h.step(u); e=h.hist[-1]
    print(f"  {e['u']:>6} {e['p']:>5} {e['g']:>5.1f}% {e['p95']:>6}ms {e['act']}")
br=sum(1 for e in h.hist if e['p95']>5000)
print(f"\n  P95 breaches: {br}/{len(h.hist)} | Final pods: {h.hist[-1]['p']}")
print(f"  Monthly: ${h.hist[-1]['p']*0.50*24*30:.0f} ({h.hist[-1]['p']} spot L4)")


</details>


---
## Exercise 3 (Medium): Hybrid Routing


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class HVR:
    def __init__(self,gmo=360):
        self.gmo=gmo; self.s={"t":0,"v":0,"a":0,"ac":0,"vq":0,"aq":0}
    def route(self,qt):
        self.s["t"]+=1
        r={"greeting":("v",0,9.5),"factual":("v",0,8.5),"pricing":("v",0,8.0),
            "summarize":("v",0,7.5),"code_simple":("v",0,8.0),
            "analysis":("a",0.003,9.0),"code_complex":("a",0.005,9.3),
            "reasoning":("a",0.004,9.0),"creative":("a",0.004,8.5)}
        d,c,q=r.get(qt,("v",0,7.5))
        if d=="v": self.s["v"]+=1; self.s["vq"]+=q
        else: self.s["a"]+=1; self.s["ac"]+=c; self.s["aq"]+=q

r=HVR(360)
qs=[]
for qt,n in [("greeting",60),("factual",80),("pricing",40),("summarize",50),("code_simple",45),
    ("analysis",40),("code_complex",25),("reasoning",35),("creative",25)]:
    qs.extend([qt]*n)
while len(qs)<500: qs.append("factual")
random.shuffle(qs); qs=qs[:500]
for q in qs: r.route(q)
s=r.s; t=max(s["t"],1)
hmo=r.gmo+s["ac"]*30; aamo=t*0.001*30; avmo=r.gmo
vq=s["vq"]/max(s["v"],1); aq=s["aq"]/max(s["a"],1); wq=(s["vq"]+s["aq"])/t

print("Hybrid vLLM + API:")
print(f"  vLLM: {s['v']} ({s['v']*100//t}%) | API: {s['a']} ({s['a']*100//t}%)")
for n,c,q in [("All API",aamo*10000/500,9.0),("Hybrid",hmo*10000/500,wq),("All vLLM",avmo,vq)]:
    print(f"  {n:<20} ${c:>7.0f}/mo Rs{c*84:>8,.0f} quality={q:.1f}")
print(f"  Savings: {(1-hmo*10000/500/(aamo*10000/500))*100:.0f}% vs all-API")


</details>


---
## Exercise 4 (Challenge): Multi-Model Serving


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

MODELS={"gemma-9b":{"q":{"greeting":9.5,"factual":9.0,"summarize":8.0,"analysis":7.5,"creative":7.0,
    "code_gen":6.5,"code_review":6.0,"debug":5.5},"cost":360},
    "qwen-coder":{"q":{"greeting":5.0,"factual":6.0,"summarize":5.5,"analysis":6.0,"creative":4.0,
    "code_gen":9.0,"code_review":8.8,"debug":8.5},"cost":360}}

def route(qt):
    m="qwen-coder" if qt in ["code_gen","code_review","debug"] else "gemma-9b"
    q=MODELS[m]["q"].get(qt,7)+random.gauss(0,0.3); return m,max(1,min(10,round(q,1)))

qs=[]
for qt,n in [("greeting",15),("factual",25),("summarize",20),("analysis",15),("creative",10),
    ("code_gen",40),("code_review",30),("debug",20)]: qs.extend([qt]*n)
qs=qs[:200]; random.shuffle(qs)

mr=[]; sr=[]
mc={"gemma-9b":0,"qwen-coder":0}; mq={"gemma-9b":0,"qwen-coder":0}
for q in qs:
    m,qual=route(q); mr.append((q,m,qual)); mc[m]+=1; mq[m]+=qual
    sq=max(1,min(10,round(MODELS["gemma-9b"]["q"].get(q,7)+random.gauss(0,0.3),1)))
    sr.append((q,sq))

print("Multi-Model Serving:")
for m in MODELS:
    if mc[m]: print(f"  {m:<16} {mc[m]:>4} req avg={mq[m]/mc[m]:.1f}/10")

print(f"\n  {'Task':<14} {'Multi':>6} {'Single':>7} {'Delta':>6}")
for task in sorted(set(qs)):
    mq_t=[q for t,_,q in mr if t==task]; sq_t=[q for t,q in sr if t==task]
    ma=sum(mq_t)/len(mq_t); sa=sum(sq_t)/len(sq_t)
    print(f"  {task:<14} {ma:>5.1f} {sa:>6.1f} {ma-sa:>+5.1f}")

om=sum(q for _,_,q in mr)/len(mr); os_=sum(q for _,q in sr)/len(sr)
code_m=[q for t,_,q in mr if t in ["code_gen","code_review","debug"]]
code_s=[q for t,q in sr if t in ["code_gen","code_review","debug"]]
print(f"\n  Overall: multi={om:.1f} single={os_:.1f} ({om-os_:+.1f})")
print(f"  Code: multi={sum(code_m)/len(code_m):.1f} single={sum(code_s)/len(code_s):.1f}"
      f" ({sum(code_m)/len(code_m)-sum(code_s)/len(code_s):+.1f})")
print(f"  Cost: multi=${720}/mo (2 GPUs) single=${360}/mo (1 GPU)")


</details>
